In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls /content/drive/MyDrive/AI_model

colab  data  model  service  src


In [ ]:
!ls /content/drive/MyDrive/AI_model/model

best_model.pkl			     logistic_regression.pkl
gaussian_nb.pkl			     naive_bayes_pipeline.joblib
k-nearest_neighbors_pipeline.joblib  scaler.pkl
knn.pkl				     support_vector_machine_pipeline.joblib
logistic_regression_pipeline.joblib  svm.pkl


In [ ]:
!mkdir -p /content/drive/MyDrive/AI_model/service/config
!mkdir -p /content/drive/MyDrive/AI_model/service/core
!mkdir -p /content/drive/MyDrive/AI_model/service/schemas
!mkdir -p /content/drive/MyDrive/AI_model/service/services
!mkdir -p /content/drive/MyDrive/AI_model/service/routers

In [ ]:
!find /content/drive/MyDrive/AI_model/service -maxdepth 2 -type d

/content/drive/MyDrive/AI_model/service
/content/drive/MyDrive/AI_model/service/config
/content/drive/MyDrive/AI_model/service/config/__pycache__
/content/drive/MyDrive/AI_model/service/core
/content/drive/MyDrive/AI_model/service/core/__pycache__
/content/drive/MyDrive/AI_model/service/schemas
/content/drive/MyDrive/AI_model/service/schemas/__pycache__
/content/drive/MyDrive/AI_model/service/services
/content/drive/MyDrive/AI_model/service/services/__pycache__
/content/drive/MyDrive/AI_model/service/routers
/content/drive/MyDrive/AI_model/service/routers/__pycache__
/content/drive/MyDrive/AI_model/service/__pycache__


In [ ]:
!touch /content/drive/MyDrive/AI_model/service/config/__init__.py
!touch /content/drive/MyDrive/AI_model/service/core/__init__.py
!touch /content/drive/MyDrive/AI_model/service/schemas/__init__.py
!touch /content/drive/MyDrive/AI_model/service/services/__init__.py
!touch /content/drive/MyDrive/AI_model/service/routers/__init__.py

In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/requirements.txt
fastapi
uvicorn
pandas
numpy
scikit-learn
joblib
pydantic

Overwriting /content/drive/MyDrive/AI_model/service/requirements.txt


In [ ]:
!cat /content/drive/MyDrive/AI_model/service/requirements.txt

fastapi
uvicorn
pandas
numpy
scikit-learn
joblib
pydantic


In [ ]:
!pip install -r /content/drive/MyDrive/AI_model/service/requirements.txt

In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/config/model_config.py
import os

# =========================
# PATH
# =========================

MODEL_DIR = "/content/drive/MyDrive/AI_model/model"

DATA_DIR = "/content/drive/MyDrive/AI_model/data"
# =========================
# ZIP DATASET
# =========================

DATASET_ZIP = "archive.zip"


# =========================
# FILE CONFIG
# =========================

SCALER_NAME = "scaler.pkl"

TARGET_NAME = "price_range"


# =========================
# FEATURES
# =========================

FEATURE_NAMES = [
    "battery_power",
    "blue",
    "clock_speed",
    "dual_sim",
    "fc",
    "four_g",
    "int_memory",
    "m_dep",
    "mobile_wt",
    "n_cores",
    "pc",
    "px_height",
    "px_width",
    "ram",
    "sc_h",
    "sc_w",
    "talk_time",
    "three_g",
    "touch_screen",
    "wifi"
]

Overwriting /content/drive/MyDrive/AI_model/service/config/model_config.py


In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/core/model_loader.py
from pathlib import Path
import joblib

from config.model_config import MODEL_DIR, SCALER_NAME


def get_model_path(model_name: str) -> Path:
    """
    Lấy đường dẫn model từ tên file.
    """

    if not model_name:
        raise ValueError("model_name không được để trống")

    if not model_name.endswith(".pkl"):
        raise ValueError("Model phải là file .pkl")

    model_dir = Path(MODEL_DIR).resolve()
    model_path = (model_dir / model_name).resolve()

    # Không cho phép truy cập ra ngoài MODEL_DIR
    if model_dir not in model_path.parents:
        raise ValueError("Đường dẫn model không hợp lệ")

    if not model_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy model: {model_name}"
        )

    return model_path


def load_model(model_name: str):
    """
    Load model theo tên file.
    """

    model_path = get_model_path(model_name)

    return joblib.load(model_path)


def get_scaler_path() -> Path:
    """
    Lấy đường dẫn scaler.
    """

    scaler_path = Path(MODEL_DIR) / SCALER_NAME

    if not scaler_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy scaler: {SCALER_NAME}"
        )

    return scaler_path


def load_scaler():
    """
    Load scaler dùng chung.
    """

    return joblib.load(get_scaler_path())


def list_models():
    """
    Trả về danh sách các model .pkl.
    Không tính scaler.pkl.
    """

    model_dir = Path(MODEL_DIR)

    if not model_dir.exists():
        raise FileNotFoundError(
            f"Không tìm thấy thư mục model: {MODEL_DIR}"
        )

    models = []

    for file in model_dir.glob("*.pkl"):
        if file.name != SCALER_NAME:
            models.append(file.name)

    return sorted(models)

Overwriting /content/drive/MyDrive/AI_model/service/core/model_loader.py


In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/schemas/mobile.py
from pydantic import BaseModel


class MobileData(BaseModel):
    battery_power: int
    blue: int
    clock_speed: float
    dual_sim: int
    fc: int
    four_g: int
    int_memory: int
    m_dep: float
    mobile_wt: int
    n_cores: int
    pc: int
    px_height: int
    px_width: int
    ram: int
    sc_h: int
    sc_w: int
    talk_time: int
    three_g: int
    touch_screen: int
    wifi: int

Overwriting /content/drive/MyDrive/AI_model/service/schemas/mobile.py


In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/services/prediction_service.py
import pandas as pd

from core.model_loader import load_model, load_scaler
from config.model_config import FEATURE_NAMES


def predict_mobile(data, model_name: str):

    model = load_model(model_name)
    scaler = load_scaler()

    input_data = pd.DataFrame(
        [data.model_dump()],
        columns=FEATURE_NAMES
    )

    # Chỉ scale đúng các cột mà scaler đã được fit (numeric_cols)
    numeric_cols = list(scaler.feature_names_in_)
    input_scaled = input_data.copy()
    input_scaled[numeric_cols] = scaler.transform(input_data[numeric_cols])

    # Đảm bảo đúng thứ tự cột như lúc train model
    input_scaled = input_scaled[FEATURE_NAMES]

    prediction = model.predict(input_scaled)

    return int(prediction[0])

Overwriting /content/drive/MyDrive/AI_model/service/services/prediction_service.py


In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/routers/prediction.py
from fastapi import APIRouter, HTTPException

from schemas.mobile import MobileData
from services.prediction_service import predict_mobile


router = APIRouter(
    prefix="/api",
    tags=["Prediction"]
)


@router.post("/predict")
def predict(
    data: MobileData,
    model: str = "best_model.pkl"
):

    try:

        prediction = predict_mobile(
            data,
            model
        )

        return {
            "model": model,
            "price_range": prediction
        }

    except FileNotFoundError as e:

        raise HTTPException(
            status_code=404,
            detail=str(e)
        )

    except ValueError as e:

        raise HTTPException(
            status_code=400,
            detail=str(e)
        )

Overwriting /content/drive/MyDrive/AI_model/service/routers/prediction.py


In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/services/evaluation_service.py
from pathlib import Path

import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

from config.model_config import DATA_DIR, FEATURE_NAMES, TARGET_NAME
from core.model_loader import load_model, load_scaler


def get_dataset_path(dataset_name: str) -> Path:
    if not dataset_name:
        raise ValueError("dataset_name không được để trống")
    if not dataset_name.endswith(".csv"):
        raise ValueError("Dataset phải là file .csv")

    data_dir = Path(DATA_DIR).resolve()

    dataset_path = None
    for path in data_dir.rglob(dataset_name):
        if path.is_file():
            dataset_path = path.resolve()
            break

    if dataset_path is None:
        raise FileNotFoundError(f"Không tìm thấy dataset: {dataset_name}")
    if data_dir not in dataset_path.parents:
        raise ValueError("Đường dẫn dataset không hợp lệ")

    return dataset_path


def evaluate_model(model_name: str, dataset_name: str):

    model = load_model(model_name)
    scaler = load_scaler()

    dataset_path = get_dataset_path(dataset_name)
    df = pd.read_csv(dataset_path)

    missing_features = [c for c in FEATURE_NAMES if c not in df.columns]
    if missing_features:
        raise ValueError(f"Dataset thiếu feature: {missing_features}")
    if TARGET_NAME not in df.columns:
        raise ValueError(f"Dataset không có cột target '{TARGET_NAME}'")

    X = df[FEATURE_NAMES].copy()
    y = df[TARGET_NAME]

    # Chỉ scale đúng các cột numeric mà scaler đã fit
    numeric_cols = list(scaler.feature_names_in_)
    X[numeric_cols] = scaler.transform(X[numeric_cols])
    X = X[FEATURE_NAMES]

    y_pred = model.predict(X)

    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y, y_pred)
    report = classification_report(y, y_pred, output_dict=True, zero_division=0)

    return {
        "model": model_name,
        "dataset": dataset_name,
        "samples": len(df),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "confusion_matrix": cm.tolist(),
        "classification_report": report
    }

Overwriting /content/drive/MyDrive/AI_model/service/services/evaluation_service.py


In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/routers/evaluation.py
from pathlib import Path
import zipfile

from fastapi import APIRouter, HTTPException

from config.model_config import DATA_DIR, DATASET_ZIP
from services.evaluation_service import evaluate_model


router = APIRouter(
    prefix="/api",
    tags=["Evaluation"]
)


def ensure_dataset_extracted():
    data_dir = Path(DATA_DIR)
    zip_path = data_dir / DATASET_ZIP

    if not zip_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file ZIP: {zip_path}")

    csv_files = list(data_dir.rglob("*.csv"))
    if csv_files:
        print("Dataset đã được giải nén.")
        return

    print(f"Đang giải nén: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(data_dir)
    print("Giải nén dataset thành công.")


@router.get("/evaluate")
def evaluate(
    model_name: str = "best_model.pkl",
    dataset_name: str = "mobile_price.csv"
):
    try:
        ensure_dataset_extracted()
        result = evaluate_model(model_name, dataset_name)
        return result

    except FileNotFoundError as e:
        raise HTTPException(status_code=404, detail=str(e))

    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))

Overwriting /content/drive/MyDrive/AI_model/service/routers/evaluation.py


In [ ]:
%%writefile /content/drive/MyDrive/AI_model/service/app.py
from fastapi import FastAPI

from routers.prediction import router as prediction_router
from routers.evaluation import router as evaluation_router

from core.model_loader import list_models


app = FastAPI(
    title="Mobile Price AI Service",
    description="AI Service for Mobile Price Classification",
    version="1.0.0"
)


# =========================
# ROUTERS
# =========================

app.include_router(
    prediction_router
)

app.include_router(
    evaluation_router
)


# =========================
# HOME
# =========================

@app.get("/")
def home():

    return {
        "service": "Mobile Price AI Service",
        "status": "running",
        "message": "AI Service is running successfully"
    }


# =========================
# HEALTH
# =========================

@app.get("/health")
def health():

    return {
        "status": "ok"
    }


# =========================
# LIST MODELS
# =========================

@app.get("/api/models")
def models():

    return {
        "models": list_models()
    }

Overwriting /content/drive/MyDrive/AI_model/service/app.py


In [ ]:
!find /content/drive/MyDrive/AI_model/service -maxdepth 2 -type f

/content/drive/MyDrive/AI_model/service/config/__init__.py
/content/drive/MyDrive/AI_model/service/config/model_config.py
/content/drive/MyDrive/AI_model/service/core/__init__.py
/content/drive/MyDrive/AI_model/service/core/model_loader.py
/content/drive/MyDrive/AI_model/service/schemas/__init__.py
/content/drive/MyDrive/AI_model/service/schemas/mobile.py
/content/drive/MyDrive/AI_model/service/services/__init__.py
/content/drive/MyDrive/AI_model/service/services/prediction_service.py
/content/drive/MyDrive/AI_model/service/services/evaluation_service.py
/content/drive/MyDrive/AI_model/service/routers/__init__.py
/content/drive/MyDrive/AI_model/service/routers/prediction.py
/content/drive/MyDrive/AI_model/service/routers/evaluation.py
/content/drive/MyDrive/AI_model/service/__pycache__/app.cpython-313.pyc
/content/drive/MyDrive/AI_model/service/requirements.txt
/content/drive/MyDrive/AI_model/service/app.py


In [ ]:
%cd /content/drive/MyDrive/AI_model/service

/content/drive/MyDrive/AI_model/service


In [ ]:
!python -c "from app import app; print('IMPORT OK')"

IMPORT OK


In [ ]:
!python -c "from core.model_loader import list_models; print(list_models())"

['best_model.pkl', 'gaussian_nb.pkl', 'knn.pkl', 'logistic_regression.pkl', 'svm.pkl']


In [ ]:
!python -c "from core.model_loader import load_model; print(type(load_model('knn.pkl')))"

<class 'sklearn.neighbors._classification.KNeighborsClassifier'>


In [ ]:
!python -c "from core.model_loader import load_model; print(type(load_model('logistic_regression.pkl')))"

<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [ ]:
!python -c "from core.model_loader import load_model; print(type(load_model('best_model.pkl')))"

<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [ ]:
%cd /content/drive/MyDrive/AI_model/service

/content/drive/MyDrive/AI_model/service


In [ ]:
!pkill -f uvicorn

In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()


def run_server():
    uvicorn.run(
        "app:app",
        host="0.0.0.0",
        port=8000
    )


thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

INFO:     Started server process [52772]
INFO:     Waiting for application startup.


## test API


In [ ]:
import requests

# This is a note/comment for the API test request.
response = requests.get(
    "http://127.0.0.1:8000/"
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:49874 - "GET / HTTP/1.1" 200 OK
200
{'service': 'Mobile Price AI Service', 'status': 'running', 'message': 'AI Service is running successfully'}


In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:49878 - "GET /health HTTP/1.1" 200 OK
200
{'status': 'ok'}


In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/api/models"
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:49888 - "GET /api/models HTTP/1.1" 200 OK
200
{'models': ['best_model.pkl', 'gaussian_nb.pkl', 'knn.pkl', 'logistic_regression.pkl', 'svm.pkl']}


In [ ]:
data = {
    "battery_power": 842,
    "blue": 0,
    "clock_speed": 2.2,
    "dual_sim": 1,
    "fc": 1,
    "four_g": 0,
    "int_memory": 7,
    "m_dep": 0.6,
    "mobile_wt": 188,
    "n_cores": 2,
    "pc": 2,
    "px_height": 20,
    "px_width": 756,
    "ram": 2549,
    "sc_h": 9,
    "sc_w": 7,
    "talk_time": 19,
    "three_g": 0,
    "touch_screen": 0,
    "wifi": 1
}

In [ ]:
response = requests.post(
    "http://127.0.0.1:8000/api/predict",
    params={
        "model": "knn.pkl"
    },
    json=data
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:49896 - "POST /api/predict?model=knn.pkl HTTP/1.1" 200 OK
200
{'model': 'knn.pkl', 'price_range': 2}


In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/api/evaluate",
    params={
        "model_name": "knn.pkl",
        "dataset_name": "train.csv"
    }
)
print(response.status_code)
print(response.json())

Dataset đã được giải nén.
INFO:     127.0.0.1:49904 - "GET /api/evaluate?model_name=knn.pkl&dataset_name=train.csv HTTP/1.1" 200 OK
200
{'model': 'knn.pkl', 'dataset': 'train.csv', 'samples': 2000, 'accuracy': 0.7205, 'precision': 0.728948356564436, 'recall': 0.7205, 'f1_score': 0.7227918481937979, 'confusion_matrix': [[415, 82, 3, 0], [99, 330, 66, 5], [12, 122, 321, 45], [1, 22, 102, 375]], 'classification_report': {'0': {'precision': 0.7874762808349146, 'recall': 0.83, 'f1-score': 0.8081791626095424, 'support': 500.0}, '1': {'precision': 0.5935251798561151, 'recall': 0.66, 'f1-score': 0.625, 'support': 500.0}, '2': {'precision': 0.6524390243902439, 'recall': 0.642, 'f1-score': 0.6471774193548387, 'support': 500.0}, '3': {'precision': 0.8823529411764706, 'recall': 0.75, 'f1-score': 0.8108108108108109, 'support': 500.0}, 'accuracy': 0.7205, 'macro avg': {'precision': 0.728948356564436, 'recall': 0.7205, 'f1-score': 0.7227918481937979, 'support': 2000.0}, 'weighted avg': {'precision': 

In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok

# Set authtoken (thay YOUR_AUTH_TOKEN bằng token thật)
ngrok.set_auth_token("3JDqtaQYGTiGRgGqBL2ShYmST87_7uYcvMQAPEEo2e8c88yaw")

# Đóng các tunnel cũ nếu có (tránh lỗi limit trên free plan)
ngrok.kill()

# Mở tunnel trỏ tới port 8000 (nơi Uvicorn đang chạy)
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://tractor-humming-pulsate.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
from core.model_loader import load_model, load_scaler
import numpy as np
import pandas as pd

model = load_model("knn.pkl")
scaler = load_scaler()

# 1. Kiểm tra scaler được fit với cột nào, theo thứ tự nào
print("Scaler feature names (nếu có):", getattr(scaler, "feature_names_in_", "Không có thông tin"))
print("Scaler mean:", getattr(scaler, "mean_", None))
print("Scaler scale (std):", getattr(scaler, "scale_", None))

# 2. Kiểm tra model biết gì về nhãn / phân phối lớp
print("Model classes:", getattr(model, "classes_", "Không rõ"))

# 3. Test scaler trực tiếp với 2 bộ dữ liệu đối lập (bỏ qua FastAPI/Node hoàn toàn)
from config.model_config import FEATURE_NAMES

# Use the feature names the scaler was fitted with (these are the numeric features)
scaled_feature_names = list(getattr(scaler, "feature_names_in_", []))

# Define raw data for low and high cases
low_data = {
    "battery_power": 501, "blue": 0, "clock_speed": 0.5, "dual_sim": 0,
    "fc": 0, "four_g": 0, "int_memory": 2, "m_dep": 0.1, "mobile_wt": 200,
    "n_cores": 1, "pc": 0, "px_height": 0, "px_width": 500, "ram": 260,
    "sc_h": 5, "sc_w": 0, "talk_time": 2, "three_g": 0, "touch_screen": 0, "wifi": 0
}
high_data = {
    "battery_power": 1999, "blue": 1, "clock_speed": 3.0, "dual_sim": 1,
    "fc": 19, "four_g": 1, "int_memory": 64, "m_dep": 1.0, "mobile_wt": 80,
    "n_cores": 8, "pc": 20, "px_height": 1960, "px_width": 1998, "ram": 3998,
    "sc_h": 19, "sc_w": 18, "talk_time": 20, "three_g": 1, "touch_screen": 1, "wifi": 1
}

# Create DataFrames with all FEATURE_NAMES
low_input_df = pd.DataFrame([low_data], columns=FEATURE_NAMES)
high_input_df = pd.DataFrame([high_data], columns=FEATURE_NAMES)

# Create copies that will be scaled and passed to the model
low_processed_for_model = low_input_df.copy()
high_processed_for_model = high_input_df.copy()

# Scale only the numeric features in these copied DataFrames
low_processed_for_model[scaled_feature_names] = scaler.transform(low_processed_for_model[scaled_feature_names])
high_processed_for_model[scaled_feature_names] = scaler.transform(high_processed_for_model[scaled_feature_names])

# Display the scaled numeric parts, matching the original print format for these values
low_numeric_scaled_values = low_processed_for_model[scaled_feature_names].values
high_numeric_scaled_values = high_processed_for_model[scaled_feature_names].values

print("\nLow scaled (numeric features only):", np.round(low_numeric_scaled_values, 2))
print("High scaled (numeric features only):", np.round(high_numeric_scaled_values, 2))

print("\nModel predict low:", model.predict(low_processed_for_model)) # Pass the full DataFrame
print("Model predict high:", model.predict(high_processed_for_model)) # Pass the full DataFrame

# 4. Đặc biệt kiểm tra cột "ram" — feature quan trọng nhất — có bị lệch scale bất thường không
ram_idx = scaled_feature_names.index("ram")
# Access ram value from the processed DataFrame for the print
print(f"\nGiá trị RAM sau scale — Low: {low_processed_for_model['ram'].iloc[0]:.2f}, High: {high_processed_for_model['ram'].iloc[0]:.2f}")

Scaler feature names (nếu có): ['battery_power' 'clock_speed' 'fc' 'int_memory' 'm_dep' 'mobile_wt'
 'n_cores' 'pc' 'px_height' 'px_width' 'ram' 'sc_h' 'sc_w' 'talk_time']
Scaler mean: [1.24080875e+03 1.51362500e+00 4.31000000e+00 3.22700000e+01
 5.02562500e-01 1.40633750e+02 4.54250000e+00 9.87812500e+00
 6.44226250e+02 1.24909188e+03 2.11613125e+03 1.22200000e+01
 5.70562500e+00 1.09568750e+01]
Scaler scale (std): [4.40589647e+02 8.19932533e-01 4.33793153e+00 1.81894777e+01
 2.86792056e-01 3.53271264e+01 2.28925616e+00 6.01296695e+00
 4.45297697e+02 4.31395297e+02 1.08037355e+03 4.20405756e+00
 4.33750716e+00 5.50602082e+00]
Model classes: [0 1 2 3]

Low scaled (numeric features only): [[-1.68 -1.24 -0.99 -1.66 -1.4   1.68 -1.55 -1.64 -1.45 -1.74 -1.72 -1.72
  -1.32 -1.63]]
High scaled (numeric features only): [[ 1.72  1.81  3.39  1.74  1.73 -1.72  1.51  1.68  2.95  1.74  1.74  1.61
   2.83  1.64]]

Model predict low: [0]
Model predict high: [3]

Giá trị RAM sau scale — Low: -1.72, H

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC # Import SVM
from sklearn.naive_bayes import GaussianNB # Import Naive Bayes

MODEL_DIR = "/content/drive/MyDrive/AI_model/model"
df = pd.read_csv("/content/drive/MyDrive/AI_model/data/train.csv")

numeric_cols = ['battery_power','clock_speed','fc','int_memory','m_dep',
                'mobile_wt','n_cores','pc','px_height','px_width','ram',
                'sc_h','sc_w','talk_time']

X = df.drop(columns=['price_range'])
y = df['price_range'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale ĐÚNG - 1 lần duy nhất, fit trên train
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols]   = scaler.transform(X_val[numeric_cols])

# Sanity check bắt buộc
assert abs(scaler.mean_[numeric_cols.index('ram')]) > 100, "Scaler vẫn sai — mean ram phải ~2000!"
print("Scaler OK, mean ram:", scaler.mean_[numeric_cols.index('ram')])

# Train lại model
knn = KNeighborsClassifier().fit(X_train_scaled, y_train)
logreg = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
svm = SVC(random_state=42).fit(X_train_scaled, y_train)
gaussian_nb = GaussianNB().fit(X_train_scaled, y_train)

print("KNN val acc:", knn.score(X_val_scaled, y_val))
print("LogReg val acc:", logreg.score(X_val_scaled, y_val))
print("SVM val acc:", svm.score(X_val_scaled, y_val))
print("Gaussian Naive Bayes val acc:", gaussian_nb.score(X_val_scaled, y_val))

# GHI ĐÈ trực tiếp vào đúng nơi service đang đọc
joblib.dump(scaler, f"{MODEL_DIR}/scaler.pkl")
joblib.dump(knn, f"{MODEL_DIR}/knn.pkl")
joblib.dump(logreg, f"{MODEL_DIR}/logistic_regression.pkl")
joblib.dump(svm, f"{MODEL_DIR}/svm.pkl")
joblib.dump(gaussian_nb, f"{MODEL_DIR}/gaussian_nb.pkl")

# Determine the best model and save it
models = {
    'knn.pkl': knn.score(X_val_scaled, y_val),
    'logistic_regression.pkl': logreg.score(X_val_scaled, y_val),
    'svm.pkl': svm.score(X_val_scaled, y_val),
    'gaussian_nb.pkl': gaussian_nb.score(X_val_scaled, y_val)
}
best_model_name = max(models, key=models.get)

if best_model_name == 'knn.pkl':
    joblib.dump(knn, f"{MODEL_DIR}/best_model.pkl")
elif best_model_name == 'logistic_regression.pkl':
    joblib.dump(logreg, f"{MODEL_DIR}/best_model.pkl")
elif best_model_name == 'svm.pkl':
    joblib.dump(svm, f"{MODEL_DIR}/best_model.pkl")
elif best_model_name == 'gaussian_nb.pkl':
    joblib.dump(gaussian_nb, f"{MODEL_DIR}/best_model.pkl")


print("✅ Đã ghi đè scaler + model mới vào", MODEL_DIR)
print(f"Best model selected: {best_model_name} with accuracy: {models[best_model_name]:.4f}")

Scaler OK, mean ram: 2116.13125
KNN val acc: 0.6
LogReg val acc: 0.98
SVM val acc: 0.91
Gaussian Naive Bayes val acc: 0.7975
✅ Đã ghi đè scaler + model mới vào /content/drive/MyDrive/AI_model/model
Best model selected: logistic_regression.pkl with accuracy: 0.9800


## Test loading SVM, Naive Bayes, and the Best Model

In [ ]:
from core.model_loader import load_model

# Load the SVM model
svm_model = load_model('svm.pkl')
print(f"Loaded SVM model type: {type(svm_model)}")

# Load the Gaussian Naive Bayes model
nb_model = load_model('gaussian_nb.pkl')
print(f"Loaded Gaussian Naive Bayes model type: {type(nb_model)}")

# Load the best model
best_model = load_model('best_model.pkl')
print(f"Loaded Best model type: {type(best_model)}")

Loaded SVM model type: <class 'sklearn.svm._classes.SVC'>
Loaded Gaussian Naive Bayes model type: <class 'sklearn.naive_bayes.GaussianNB'>
Loaded Best model type: <class 'sklearn.linear_model._logistic.LogisticRegression'>


In [ ]:
response = requests.post(
    "http://127.0.0.1:8000/api/predict",
    params={
        "model": "svm.pkl" # Specify your SVM model here
    },
    json=data # Use your mobile data here
)
print(response.status_code)
print(response.json())

INFO:     127.0.0.1:35622 - "POST /api/predict?model=svm.pkl HTTP/1.1" 200 OK
200
{'model': 'svm.pkl', 'price_range': 1}


In [ ]:
response = requests.post(
    "http://127.0.0.1:8000/api/predict",
    params={
        "model": "gaussian_nb.pkl" # Specify your Naive Bayes model here
    },
    json=data # Use your mobile data here
)
print(response.status_code)
print(response.json())

INFO:     127.0.0.1:35634 - "POST /api/predict?model=gaussian_nb.pkl HTTP/1.1" 200 OK
200
{'model': 'gaussian_nb.pkl', 'price_range': 2}


In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/api/evaluate",
    params={
        "model_name": "svm.pkl", # Specify your SVM model here
        "dataset_name": "train.csv"
    }
)
print(response.status_code)
print(response.json())

Dataset đã được giải nén.
INFO:     127.0.0.1:35640 - "GET /api/evaluate?model_name=svm.pkl&dataset_name=train.csv HTTP/1.1" 200 OK
200
{'model': 'svm.pkl', 'dataset': 'train.csv', 'samples': 2000, 'accuracy': 0.9685, 'precision': 0.9687865542131181, 'recall': 0.9685, 'f1_score': 0.9685909519847851, 'confusion_matrix': [[490, 10, 0, 0], [8, 481, 11, 0], [0, 17, 480, 3], [0, 0, 14, 486]], 'classification_report': {'0': {'precision': 0.9839357429718876, 'recall': 0.98, 'f1-score': 0.9819639278557114, 'support': 500.0}, '1': {'precision': 0.9468503937007874, 'recall': 0.962, 'f1-score': 0.9543650793650794, 'support': 500.0}, '2': {'precision': 0.9504950495049505, 'recall': 0.96, 'f1-score': 0.9552238805970149, 'support': 500.0}, '3': {'precision': 0.9938650306748467, 'recall': 0.972, 'f1-score': 0.9828109201213346, 'support': 500.0}, 'accuracy': 0.9685, 'macro avg': {'precision': 0.968786554213118, 'recall': 0.9685, 'f1-score': 0.9685909519847851, 'support': 2000.0}, 'weighted avg': {'pre

In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/api/evaluate",
    params={
        "model_name": "svm.pkl", # Specify your SVM model here
        "dataset_name": "train.csv"
    }
)
print(response.status_code)
print(response.json())

Dataset đã được giải nén.
INFO:     127.0.0.1:35642 - "GET /api/evaluate?model_name=svm.pkl&dataset_name=train.csv HTTP/1.1" 200 OK
200
{'model': 'svm.pkl', 'dataset': 'train.csv', 'samples': 2000, 'accuracy': 0.9685, 'precision': 0.9687865542131181, 'recall': 0.9685, 'f1_score': 0.9685909519847851, 'confusion_matrix': [[490, 10, 0, 0], [8, 481, 11, 0], [0, 17, 480, 3], [0, 0, 14, 486]], 'classification_report': {'0': {'precision': 0.9839357429718876, 'recall': 0.98, 'f1-score': 0.9819639278557114, 'support': 500.0}, '1': {'precision': 0.9468503937007874, 'recall': 0.962, 'f1-score': 0.9543650793650794, 'support': 500.0}, '2': {'precision': 0.9504950495049505, 'recall': 0.96, 'f1-score': 0.9552238805970149, 'support': 500.0}, '3': {'precision': 0.9938650306748467, 'recall': 0.972, 'f1-score': 0.9828109201213346, 'support': 500.0}, 'accuracy': 0.9685, 'macro avg': {'precision': 0.968786554213118, 'recall': 0.9685, 'f1-score': 0.9685909519847851, 'support': 2000.0}, 'weighted avg': {'pre

In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/api/evaluate",
    params={
        "model_name": "gaussian_nb.pkl", # Specify your Naive Bayes model here
        "dataset_name": "train.csv"
    }
)
print(response.status_code)
print(response.json())

Dataset đã được giải nén.
INFO:     127.0.0.1:35658 - "GET /api/evaluate?model_name=gaussian_nb.pkl&dataset_name=train.csv HTTP/1.1" 200 OK
200
{'model': 'gaussian_nb.pkl', 'dataset': 'train.csv', 'samples': 2000, 'accuracy': 0.815, 'precision': 0.8173882616170098, 'recall': 0.815, 'f1_score': 0.8160276512861075, 'confusion_matrix': [[450, 50, 0, 0], [39, 370, 91, 0], [0, 79, 372, 49], [0, 1, 61, 438]], 'classification_report': {'0': {'precision': 0.9202453987730062, 'recall': 0.9, 'f1-score': 0.910010111223458, 'support': 500.0}, '1': {'precision': 0.74, 'recall': 0.74, 'f1-score': 0.74, 'support': 500.0}, '2': {'precision': 0.7099236641221374, 'recall': 0.744, 'f1-score': 0.7265625, 'support': 500.0}, '3': {'precision': 0.8993839835728953, 'recall': 0.876, 'f1-score': 0.8875379939209727, 'support': 500.0}, 'accuracy': 0.815, 'macro avg': {'precision': 0.8173882616170097, 'recall': 0.8150000000000001, 'f1-score': 0.8160276512861077, 'support': 2000.0}, 'weighted avg': {'precision': 0.